In [12]:
import os
from sklearn.feature_extraction.text import HashingVectorizer
# from sklearn.linear_model import SGDClassifier
from sklearn import metrics
# vectorizer = HashingVectorizer()
from pathlib import Path
# classifier = SGDClassifier(warm_start=False)
import joblib
import numpy as np
from transformers import AutoTokenizer, AutoModel
# from torch import no_grad
import torch
from nn_model import Model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased', attn_implementation = "eager")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 855.76it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
def save_model(classifier, model_path):
    joblib.dump(classifier, os.path.join(model_path, 'classifier.pkl'))
    # joblib.dump(tokenizer, os.path.join(model_path, 'tokenizer.pkl'))
    # joblib.dump(vectorizer, os.path.join(model_path, 'vectorizer.pkl'))

In [14]:
def load_model(model_path):
    classifier = joblib.load(os.path.join(model_path, 'classifier.pkl'))
    # tokenizer = joblib.load(os.path.join(model_path, 'tokenizer.pkl'))
    # vectorizer = joblib.load(os.path.join(model_path, 'vectorizer.pkl'))
    return classifier, tokenizer

In [15]:
def get_bert_embeddings(text):
    # Tokenize the text
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True, attention="eager")
    
    # Get the last hidden state (contextual embeddings)
    embeddings = outputs.last_hidden_state  # Shape: [seq_len, hidden_dim]
    return embeddings

In [16]:
def get_many_bert_embeddings(texts):
    embeddings = []
    for text in texts:
        embedding = get_bert_embeddings(text)
        embeddings.append((embedding))
    return embeddings

In [ ]:
def train_model(training_path=None, transfer_learning=False, model_path=None, author_notes_pairs=None):
    if not training_path and not author_notes_pairs:
        raise ValueError("Either 'training_path' or 'author_notes_pairs' must be provided")
    if transfer_learning and model_path:
        old_model = load_model(model_path)
    corpus = []
    embeddings = []
    authors_per_document = []
    authors = []
    index = 0
    if training_path:
        for file in os.listdir(training_path):
            if (Path(os.path.join(training_path, file))).is_dir():
                index += 1
                if (index <= 3 and not transfer_learning) or (index > 24 and transfer_learning):
                    authors.append(file)
                    for fileName in os.listdir(os.path.join(training_path, file)):
                        with open(os.path.join(training_path, file, fileName), 'r') as f:
                            text = f.read()
                            corpus.append(text.lower())
                            authors_per_document.append(file)
    else:
        for author, text in author_notes_pairs:
            corpus.append(text.lower())
            authors_per_document.append(author)
    embeddings = get_many_bert_embeddings(corpus)
    author_to_int = {author: i for i, author in enumerate(set(authors))}
    if transfer_learning and model_path:
        pass
    else:
        model = Model(embedding_dim=index)
        device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
        model.to(device)
        num_classes = index
        model.train()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        loss_fn = torch.nn.MSELoss()
        for epoch in range(100):
            total_loss = 0
            model.train()
            for i in range(len(authors_per_document)):
                author = authors_per_document[i]
                author_int = author_to_int[author]
                author_torch = torch.tensor([author_int], dtype=torch.long).to(device)
                embedding = embeddings[i]
                embedding = embedding.to(device)

                optimizer.zero_grad()
                prediction = model(embedding)
                loss = loss_fn(prediction, author_torch)

                loss.backward()
                optimizer.step()

                total_loss += loss.item()
        
    predictions = []
    for i in range(len(embeddings)):
        model.eval()
        embedding = embeddings[i]
        embedding = embedding.to(device)
        with torch.no_grad():
            prediction = model(embedding)
        predictions.append(prediction.argmax().item())
    f1 = metrics.f1_score([author_to_int[author] for author in authors_per_document], predictions, average='macro')
    print(f"F1 Score: {f1}")
    # score = classifier.score(embeddings, authors_per_document)
    # print(f"Accuracy: {score}")
    return model

In [18]:
model = train_model('./reuter+50+50/C50train/')

TypeError: Model.__init__() got an unexpected keyword argument 'class_amount'

In [ ]:
def test_model(classifier, testing_path, transfer_learning=False):
    test_corpus = []
    test_authors = []
    test_embeddings = []
    index = 0
    for file in os.listdir(testing_path):
        if (Path(os.path.join(testing_path, file))).is_dir():
            index += 1
            if (index <= 24 and not transfer_learning) or (index > 24 and transfer_learning):
                for fileName in os.listdir(os.path.join(testing_path, file)):
                    with open(os.path.join(testing_path, file, fileName), 'r') as f:
                        text = f.read()
                        test_corpus.append(text.lower())
                        test_authors.append(file)
    test_embeddings = get_many_bert_embeddings(test_corpus)
    # test_matrix = vectorizer.transform(test_corpus)
    predictions = []
    for i, embedding in enumerate(test_embeddings):
        predicted_author = classifier.predict(embedding)
        predictions.append(predicted_author[0])
    f1 = metrics.f1_score(test_authors, predictions, average='macro')
    print(f"F1 Score: {f1}")
    score = classifier.score(test_embeddings, test_authors)
    print(f"Accuracy: {score}")

In [ ]:
# test_model(classifier, vectorizer, './reuter+50+50/C50test/')

In [ ]:
save_model(model, './models/')

In [ ]:
classifier, vectorizer = train_model('./reuter+50+50/C50train/', transfer_learning=True, model_path='./models/')

TypeError: Model.forward() got an unexpected keyword argument 'input_ids'

In [ ]:
# test_model(classifier, vectorizer, './reuter+50+50/C50test/', transfer_learning=True)

F1 Score: 0.792177507749996
Accuracy: 0.7930769230769231


In [ ]:
# classifier, vectorizer = load_model('./models/')

In [ ]:
# model.eval()
        # with torch.no_grad():
        #     for batch in test_loader:
        #         first, second, classes = batch
        #         first, second, classes = first.to(device), second.to(device), classes.to(device)
        #         em_first = model(first)
        #         em_second = model(second)
        #         similarity = F.cosine_similarity(em_first, em_second, dim=1)
        #         isMatch = classes[:, 0] == classes[:, 1]

        #         predictions = similarity > 0.7
        #         predictions_high = similarity > 0.8
        #         total_correct += torch.sum(predictions == isMatch).item()
        #         total_correct_high += torch.sum(predictions_high == isMatch).item()

        #         total_similar += torch.sum(isMatch).item()
        #         total_opposing += torch.sum(~isMatch).item()

        #         total_similarity_match += torch.sum(similarity[isMatch]).item()
        #         total_similarity_diff += torch.sum(similarity[~isMatch]).item()

        # training_distribution_similar = count_similar_correct_per_class / count_similar_per_class
        # training_distribution_opposing = count_opposing_correct_per_class / count_opposing_per_class

        # reversed_training_distribution_similar = 1 - training_distribution_similar
        # reversed_training_distribution_opposing = 1 - training_distribution_opposing

        # reversed_training_distribution_similar = torch.nan_to_num(reversed_training_distribution_similar.cpu(), 0.5)
        # reversed_training_distribution_opposing = torch.nan_to_num(reversed_training_distribution_opposing.cpu().fill_diagonal_(0), 0.5)

        # probability_distribution_similar = torch.softmax(reversed_training_distribution_similar * 7, dim=0)
        # probability_distribution_opposing = torch.softmax(reversed_training_distribution_opposing.flatten() * 7, dim=0).reshape(num_classes, num_classes)


        # distributionSimilar = probability_distribution_similar
        # distributionOpposing = probability_distribution_opposing